# Attempt 3: Slope Map Line Tracing via Tiled Molmo2

**Goal:** Trace hydraulic feature lines (channel banks, road embankments, terrace edges) directly from the DEM slope map by tiling at full resolution and feeding to Molmo2.

**Pipeline:** Tile slope map → Molmo2 traces dark lines per tile → stitch → classify with Gemma4 → georeferenced feature polylines

## Approach

The slope map already shows all breaklines as dark/red lines. Instead of asking a VLM to understand hydraulics from satellite imagery, we simply ask Molmo2 to **trace visible lines** in the slope map. This is a fundamentally easier task: the model only needs to identify and follow dark linear features, not interpret what they represent hydrologically.

Classification of the traced lines (channel bank vs. road embankment vs. terrace edge) is deferred to a second pass where Gemma4 can use the satellite imagery for context.

## Pipeline Stages

| Stage | Input | Process | Output |
|-------|-------|---------|--------|
| 1. Tile | Slope map (full res PNG) | Split into overlapping 1024×1024 tiles | List of tile images + offsets |
| 2. Trace | Each tile | Molmo2 pointing: "trace dark lines" | Point coordinates per tile |
| 3. Stitch | All tile points | Convert to global pixel coords, deduplicate overlap | Unified point cloud |
| 4. Classify | Points + satellite | Gemma4 labels each line segment by feature type | Typed polylines |
| 5. Export | Typed polylines + DEM CRS | Convert pixel → geo coords, write to GeoJSON/shapefile | Georeferenced feature lines |

**Notebook scope:** Stages 1–3 (tile, trace, stitch). Classification and export are deferred to subsequent notebooks.

In [ ]:
import os
import re
import json
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import rasterio
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

# Paths
DEM_PATH = Path("../../data/input/1m elevation.tif")
SLOPE_PATH = Path("../../data/output/cimarron_slope.png")
SATELLITE_PATH = Path("../../data/output/cimarron_satellite.png")
OUTPUT_DIR = Path("../../data/output/model-outputs/attempt3/slope-tracing/")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Timestamp for this run
RUN_TS = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

print(f"DEM: {DEM_PATH}")
print(f"Slope map: {SLOPE_PATH}")
print(f"Satellite image: {SATELLITE_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Run timestamp: {RUN_TS}")
print(f"PyTorch version: {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 1.1 Load Molmo 2

In [ ]:
from transformers import QuantoConfig

MODEL_ID = "allenai/Molmo2-8B"
MOLMO_AVAILABLE = False

device = "mps" if torch.backends.mps.is_available() else "cpu"

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

    quantization_config = QuantoConfig(weights="int4")
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=quantization_config,
        torch_dtype=torch.float16,
    ).to(device)
    model.eval()

    # Quick sanity check (text-only)
    test_inputs = processor.apply_chat_template(
        [{"role": "user", "content": [dict(type="text", text="Say hello in one word.")]}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    test_inputs = {k: v.to(device) for k, v in test_inputs.items()}
    with torch.no_grad():
        test_out = model.generate(**test_inputs, max_new_tokens=10)
    test_response = processor.tokenizer.decode(
        test_out[0, test_inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(f"Molmo 2 loaded OK: {test_response}")
    print(f"Model: {MODEL_ID} (Quanto int4)")
    print(f"Device: {device}")
    MOLMO_AVAILABLE = True
except Exception as e:
    print(f"Molmo 2 failed to load.")
    print(f"Error: {e}")
    import traceback; traceback.print_exc()
    print()
    print("Setup instructions:")
    print("  1. pip install transformers==4.57.1 accelerate einops molmo_utils optimum-quanto")
    print("  2. Ensure ~6 GB free disk space for model download")
    print("  3. Requires PyTorch with MPS support (Apple Silicon) or CUDA")
    print()
    print("The rest of this notebook will be skipped.")

## 1.2 Load Slope Map & DEM Metadata

In [ ]:
# Load slope map
if not SLOPE_PATH.exists():
    raise FileNotFoundError(
        f"Slope map not found at {SLOPE_PATH}.\n"
        "Generate the slope map from the DEM first."
    )

img_slope = Image.open(SLOPE_PATH)
print(f"Slope map: {img_slope.size} ({img_slope.mode})")

# Load DEM metadata for geo-coordinate conversion
with rasterio.open(DEM_PATH) as src:
    dem_transform = src.transform
    dem_bounds = src.bounds
    dem_crs = src.crs

print(f"CRS: {dem_crs}")
print(f"Bounds: {dem_bounds}")

# Load satellite image for later comparison
img_satellite = None
if SATELLITE_PATH.exists():
    img_satellite = Image.open(SATELLITE_PATH)
    print(f"Satellite image: {img_satellite.size} ({img_satellite.mode})")

# Display the slope map
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(img_slope)
ax.set_title("Slope Map (dark lines = breaklines)", fontsize=14)
ax.set_xlabel(f"{img_slope.size[0]} x {img_slope.size[1]} px")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 1.3 Tiling Utilities

In [ ]:
def create_tiles(img, tile_size=1024, overlap=128):
    """Split image into overlapping tiles.

    Returns list of (tile_image, x_offset, y_offset) tuples.
    Skips tiles that are mostly empty (uniform color).
    """
    tiles = []
    w, h = img.size
    step = tile_size - overlap

    for y in range(0, h, step):
        for x in range(0, w, step):
            # Crop tile (PIL handles edge cases automatically)
            box = (x, y, min(x + tile_size, w), min(y + tile_size, h))
            tile = img.crop(box)

            # Skip mostly-empty tiles (low variance = flat terrain)
            arr = np.array(tile)
            if arr.std() < 10:  # threshold for "empty" tile
                continue

            tiles.append((tile, x, y))

    return tiles


def tile_points_to_global(points_px, x_offset, y_offset):
    """Convert tile-local pixel coordinates to full-image coordinates."""
    return [(px + x_offset, py + y_offset, label) for px, py, label in points_px]


def deduplicate_points(all_points, min_dist=10):
    """Remove duplicate points from overlapping tile regions."""
    if not all_points:
        return []

    unique = [all_points[0]]
    for px, py, label in all_points[1:]:
        is_dup = False
        for ux, uy, _ in unique:
            if (px - ux)**2 + (py - uy)**2 < min_dist**2:
                is_dup = True
                break
        if not is_dup:
            unique.append((px, py, label))
    return unique


# Preview: how many tiles would be created?
preview_tiles = create_tiles(img_slope, tile_size=1024, overlap=128)
print(f"Slope map size: {img_slope.size}")
print(f"Tile size: 1024x1024, overlap: 128")
print(f"Tiles created (after skipping empty): {len(preview_tiles)}")

# Show tile grid
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.imshow(img_slope)
for tile_img, x_off, y_off in preview_tiles:
    tw, th = tile_img.size
    rect = plt.Rectangle((x_off, y_off), tw, th,
                          linewidth=0.5, edgecolor='cyan', facecolor='none', alpha=0.5)
    ax.add_patch(rect)
ax.set_title(f"Tile Grid ({len(preview_tiles)} tiles)", fontsize=14)
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## 1.4 Molmo2 Inference & Parsing

In [ ]:
def vision_query(image, prompt, max_new_tokens=512):
    """Send an image + text prompt to Molmo 2 via HuggingFace Transformers."""
    messages = [
        {
            "role": "user",
            "content": [
                dict(type="text", text=prompt),
                dict(type="image", image=image),
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    # Decode only the generated tokens (skip the input prompt)
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    return processor.tokenizer.decode(generated_ids, skip_special_tokens=True)


# Molmo2 coordinate parsing regexes (from official docs)
COORD_REGEX = re.compile(r'<(?:points|tracks)[^>]* coords="([^"]+)"')
POINTS_REGEX = re.compile(r'([0-9]+) ([0-9]{3,4}) ([0-9]{3,4})')


def parse_molmo_points(text, image_w, image_h):
    """Parse Molmo2 coordinate output into pixel coordinates.

    Molmo2 outputs coords as triplets: index x y
    where x,y are 3-4 digit integers on a 0-1000 scale.

    Also handles legacy <point x="..." y="..."> format (0-100 scale).

    Returns list of (x_px, y_px, label) tuples.
    """
    points = []

    # Molmo2 format: <points coords="idx x y idx x y ...">label</points>
    for coord_match in COORD_REGEX.finditer(text):
        coord_str = coord_match.group(1)
        for pt_match in POINTS_REGEX.finditer(coord_str):
            idx = pt_match.group(1)
            x_raw, y_raw = float(pt_match.group(2)), float(pt_match.group(3))
            x_px = x_raw / 1000.0 * image_w
            y_px = y_raw / 1000.0 * image_h
            if 0 <= x_px <= image_w and 0 <= y_px <= image_h:
                points.append((x_px, y_px, f"point_{idx}"))

    # Legacy format: <point x="..." y="...">label</point> (0-100 scale)
    if not points:
        legacy_pattern = r'<point\s+x="([\d.]+)"\s+y="([\d.]+)"(?:\s+alt="[^"]*")?>([^<]*)</point>'
        for match in re.finditer(legacy_pattern, text):
            x_pct, y_pct, label = float(match.group(1)), float(match.group(2)), match.group(3).strip()
            x_px = x_pct / 100.0 * image_w
            y_px = y_pct / 100.0 * image_h
            points.append((x_px, y_px, label))

    return points


def pixels_to_geo(points, transform):
    """Convert pixel coordinates to geo-coordinates using rasterio transform."""
    geo_coords = []
    for x_px, y_px, label in points:
        easting, northing = rasterio.transform.xy(transform, int(y_px), int(x_px))
        geo_coords.append((easting, northing, label))
    return geo_coords


def trace_tile(tile_img, prompt, tile_w=None, tile_h=None):
    """Run a line-tracing query on a single tile and parse the results.

    Returns dict with raw_output, points (pixel coords), and count.
    """
    if tile_w is None:
        tile_w = tile_img.size[0]
    if tile_h is None:
        tile_h = tile_img.size[1]

    raw = vision_query(tile_img, prompt)
    points = parse_molmo_points(raw, tile_w, tile_h)
    return {
        "raw_output": raw,
        "points": points,
        "n_points": len(points),
    }


print("Inference and parsing utilities ready.")

## 2.1 Trace Dark Lines — Full Tiled Run

In [ ]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available.")
else:
    PROMPT = "Trace all dark lines visible in this image. Place points along each dark line."

    tiles = create_tiles(img_slope, tile_size=1024, overlap=128)
    print(f"Processing {len(tiles)} tiles...")

    all_points = []
    tile_results = []

    for i, (tile_img, x_off, y_off) in enumerate(tiles):
        print(f"\nTile {i+1}/{len(tiles)} (offset: {x_off}, {y_off}, size: {tile_img.size})")

        raw = vision_query(tile_img, PROMPT)
        tile_w, tile_h = tile_img.size
        points_local = parse_molmo_points(raw, tile_w, tile_h)
        points_global = tile_points_to_global(points_local, x_off, y_off)

        tile_results.append({
            "tile_idx": i,
            "offset": (x_off, y_off),
            "size": tile_img.size,
            "raw_output": raw,
            "n_points": len(points_local),
            "points_local": points_local,
            "points_global": points_global,
        })

        all_points.extend(points_global)
        print(f"  Points found: {len(points_local)}")
        print(f"  Raw: {raw[:100]}...")

    # Deduplicate overlapping points
    unique_points = deduplicate_points(all_points)
    print(f"\nTotal raw points: {len(all_points)}")
    print(f"After dedup: {len(unique_points)}")

    # Convert to geo-coordinates
    geo_points = pixels_to_geo(unique_points, dem_transform)

## 3.1 Visualization & Results

In [ ]:
if not MOLMO_AVAILABLE:
    print("Skipping — Molmo 2 is not available. No results to visualize.")
else:
    # --- Plot points on slope map ---
    fig, ax = plt.subplots(1, 1, figsize=(16, 10))
    ax.imshow(img_slope)
    if unique_points:
        xs = [p[0] for p in unique_points]
        ys = [p[1] for p in unique_points]
        ax.scatter(xs, ys, c="red", s=4, marker=".", alpha=0.6, zorder=5)
    ax.set_title(f"Traced Lines on Slope Map ({len(unique_points)} points)", fontsize=14)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.tight_layout()
    slope_overlay_path = OUTPUT_DIR / f"{RUN_TS}_slope_traced_points.png"
    plt.savefig(slope_overlay_path, dpi=150, bbox_inches="tight")
    print(f"Saved slope overlay: {slope_overlay_path}")
    plt.show()

    # --- Plot points on satellite image (for comparison) ---
    if img_satellite is not None:
        fig, ax = plt.subplots(1, 1, figsize=(16, 10))
        ax.imshow(img_satellite)
        if unique_points:
            # Scale points from slope map coordinates to satellite coordinates
            slope_w, slope_h = img_slope.size
            sat_w, sat_h = img_satellite.size
            xs_sat = [p[0] * sat_w / slope_w for p in unique_points]
            ys_sat = [p[1] * sat_h / slope_h for p in unique_points]
            ax.scatter(xs_sat, ys_sat, c="red", s=4, marker=".", alpha=0.6, zorder=5)
        ax.set_title(f"Traced Lines on Satellite ({len(unique_points)} points)", fontsize=14)
        ax.set_xticks([])
        ax.set_yticks([])
        plt.tight_layout()
        sat_overlay_path = OUTPUT_DIR / f"{RUN_TS}_satellite_traced_points.png"
        plt.savefig(sat_overlay_path, dpi=150, bbox_inches="tight")
        print(f"Saved satellite overlay: {sat_overlay_path}")
        plt.show()

    # --- Save geo-coordinates to JSON ---
    geo_json_data = {
        "run_timestamp": RUN_TS,
        "model": MODEL_ID,
        "crs": str(dem_crs),
        "prompt": PROMPT,
        "tile_size": 1024,
        "overlap": 128,
        "n_tiles": len(tiles),
        "n_raw_points": len(all_points),
        "n_unique_points": len(unique_points),
        "points": [
            {"easting": e, "northing": n, "label": l}
            for e, n, l in geo_points
        ],
        "tile_results": [
            {
                "tile_idx": r["tile_idx"],
                "offset": r["offset"],
                "size": r["size"],
                "n_points": r["n_points"],
                "raw_output": r["raw_output"],
            }
            for r in tile_results
        ],
    }

    json_path = OUTPUT_DIR / f"{RUN_TS}_slope_traced_geo.json"
    with open(json_path, "w") as f:
        json.dump(geo_json_data, f, indent=2)
    print(f"Saved geo-coordinates: {json_path}")

    # --- Save markdown results summary ---
    md_lines = [
        f"# Slope Map Line Tracing Results\n",
        f"\n**Date:** {RUN_TS}\n",
        f"\n**Model:** `{MODEL_ID}` (Quanto int4, HuggingFace Transformers)\n",
        f"\n**Input:** Slope map ({img_slope.size[0]}x{img_slope.size[1]})\n",
        f"\n**Prompt:** {PROMPT}\n",
        f"\n**CRS:** {dem_crs}\n",
        f"\n---\n",
        f"\n## Tiling\n",
        f"\n- Tile size: 1024x1024\n",
        f"- Overlap: 128 px\n",
        f"- Tiles processed: {len(tiles)}\n",
        f"\n## Results\n",
        f"\n- Raw points (all tiles): {len(all_points)}\n",
        f"- Unique points (after dedup): {len(unique_points)}\n",
        f"- Deduplication distance: 10 px\n",
        f"\n## Per-Tile Breakdown\n",
        f"\n| Tile | Offset | Size | Points |\n",
        f"|------|--------|------|--------|\n",
    ]
    for r in tile_results:
        md_lines.append(
            f"| {r['tile_idx']} | ({r['offset'][0]}, {r['offset'][1]}) "
            f"| {r['size'][0]}x{r['size'][1]} | {r['n_points']} |\n"
        )

    md_path = OUTPUT_DIR / f"{RUN_TS}_slope_tracing_results.md"
    with open(md_path, "w") as f:
        f.writelines(md_lines)
    print(f"Saved results summary: {md_path}")

    # --- Print summary ---
    print(f"\n{'='*60}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"Tiles processed:   {len(tiles)}")
    print(f"Raw points:        {len(all_points)}")
    print(f"Unique points:     {len(unique_points)}")
    print(f"Geo-coords saved:  {json_path}")
    print(f"Slope overlay:     {slope_overlay_path}")
    if img_satellite is not None:
        print(f"Satellite overlay: {sat_overlay_path}")
    print(f"Results markdown:  {md_path}")

## 4. Next Steps

With the raw traced points from the slope map, the remaining pipeline stages are:

1. **Classify traced lines using Gemma4 + satellite imagery** — For each cluster of traced points, crop the corresponding satellite patch and ask Gemma4 to identify the feature type (channel bank, road embankment, terrace edge, field boundary, etc.).

2. **Group points into polylines (connected feature lines)** — Use spatial clustering (e.g., DBSCAN or nearest-neighbor chaining) to connect individual traced points into coherent polyline segments that follow the dark lines in the slope map.

3. **Assign vertex spacing per feature type** — Different hydraulic features require different vertex densities: channel banks need dense vertices at meander bends, while straight road embankments can use sparser spacing.

4. **Export as georeferenced feature lines for SMS/HEC-RAS 2D** — Convert the typed, connected polylines to GeoJSON or shapefile format with proper CRS metadata, ready for import into hydraulic modeling software.